# Sonata + VGC on Colab — Conda Environment (Matches Official Setup)Uses **condacolab** to install miniconda, then **environment.yml** to recreatethe EXACT same environment as the official Pointcept/Sonata repository.**No more version conflicts.** Python 3.10, PyTorch 2.5.0, CUDA 12.4, spconv-cu124 all locked.> After Cell 1 completes, Colab will prompt to **restart the runtime**.> Click Restart, then continue from Cell 2.

In [4]:
# ═══════════════════════════════════════
# CELL 1 — Install Miniconda + Create Env
# ═══════════════════════════════════════
import subprocess, os, sys

# 1. Install condacolab (gets miniconda on Colab)!
!pip install -q condacolab
import condacolab
condacolab.install()

# Ensure we are in /content before cloning to avoid 'shell-init' errors
%cd /content

# 2. Clone repo (need environment.yml)!
!rm -rf /content/voxel_group_classifier
!git clone --depth 1 https://github.com/yuhang-wang-xjtu/voxel_group_classifier.git /content/voxel_group_classifier

# Change to the cloned directory for environment creation and other operations
%cd /content/voxel_group_classifier

# 3. Create conda env from official environment.yml
#    This installs: python=3.10, pytorch=2.5.0, cuda=12.4, spconv-cu124, torch-cluster, etc.

# Define environment name and repository directory for clarity
ENV_NAME = "pointcept-torch2.5.0-cu12.4"
REPO_DIR = "/content/voxel_group_classifier"

# Remove existing environment if it exists (use '|| true' to prevent error if it doesn't exist)
!conda env remove --name {ENV_NAME} --yes || true
# Create the environment, explicitly providing the full path to environment.yml
!conda env create -f {REPO_DIR}/environment.yml

# 4. Register the env as a Colab kernel!
# Ensure pip is installed in the new environment first (already done by environment.yml, but this is a good safety check)
!conda run -n {ENV_NAME} pip install ipykernel
# Register the kernel, explicitly running within the new conda environment
!conda run -n {ENV_NAME} python -m ipykernel install --user --name=pointcept --display-name="Sonata (Python 3.10)"

print("\n" + "="*60)
print("Environment created. Restart the runtime NOW.")
print("After restart, select kernel: 'Sonata (Python 3.10)' from Runtime → Change runtime type")
print("Then continue with Cell 2.")

✨🍰✨ Everything looks OK!
/content
Cloning into '/content/voxel_group_classifier'...
remote: Enumerating objects: 720, done.
remote: Counting objects: 100% (720/720), done.
remote: Compressing objects: 100% (369/369), done.
remote: Total 720 (delta 336), reused 716 (delta 336), pack-reused 0 (from 0)
Receiving objects: 100% (720/720), 4.73 MiB | 9.37 MiB/s, done.
Resolving deltas: 100% (336/336), done.
/content/voxel_group_classifier

Remove all packages in environment /usr/local/envs/pointcept-torch2.5.0-cu12.4:


## Package Plan ##

  environment location: /usr/local/envs/pointcept-torch2.5.0-cu12.4


The following packages will be REMOVED:

  _openmp_mutex-4.5-20_gnu
  _python_abi3_support-1.0-hd8ed1ab_3
  absl-py-2.5.0-pyhd8ed1ab_0
  accelerate-1.14.0-pyhcf101f3_0
  addict-2.4.0-pyhcf101f3_4
  aiohappyeyeballs-2.7.1-pyhd8ed1ab_0
  aiohttp-3.14.2-py310h31b6992_0
  aiosignal-1.4.0-pyhd8ed1ab_0
  alsa-lib-1.2.16.1-hb03c661_0
  annotated-doc-0.0.4-pyhcf101f3_0
  annotated-types-0.7.0-py

In [1]:
import sys
import os

print(f"Current Python executable: {sys.executable}")
print(f"Current Python version: {sys.version}")
print(f"Current CONDA_PREFIX: {os.environ.get('CONDA_PREFIX', 'Not set')}")

try:
    import spconv
    print(f"spconv successfully imported! Version: {spconv.__version__}")
except ModuleNotFoundError:
    print("Error: spconv module not found. The 'Sonata (Python 3.10)' kernel might not be active.")
    print("Please ensure you have restarted the runtime and selected 'Sonata (Python 3.10)' from Runtime → Change runtime type.")


Current Python executable: /usr/bin/python3.real
Current Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Current CONDA_PREFIX: Not set
Error: spconv module not found. The 'Sonata (Python 3.10)' kernel might not be active.
Please ensure you have restarted the runtime and selected 'Sonata (Python 3.10)' from Runtime → Change runtime type.


In [6]:
# IMI IMI IMI IMI IMI IMI IMI IMI IMI IMI IMI IMI
# CELL 2 - Verify Environment (Auto-Fallback)
# IMI IMI IMI IMI IMI IMI IMI IMI IMI IMI IMI IMI
# Run AFTER Colab restart. If kernel is still system Python, this cell
# auto-falls back to conda run instead of requiring manual kernel switch.

ENV_NAME = "pointcept-torch2.5.0-cu12.4"
ENV_PYTHON = f"/usr/local/envs/{ENV_NAME}/bin/python"

import os, sys, subprocess

# Detect if we're in the right conda env
in_conda = "conda" in sys.executable or "envs/pointcept" in sys.executable
print(f"Current Python: {sys.executable}")
print(f"Version: {sys.version.split()[0]}")
print(f"CONDA_PREFIX: {os.environ.get('CONDA_PREFIX', 'Not set')}")

if not in_conda and not os.path.exists(ENV_PYTHON):
    print("ERROR: Conda env not found. Re-run Cell 1.")
    raise SystemExit(1)

if not in_conda:
    print(f"WARNING: Not in conda env. Using '{ENV_PYTHON}' for all commands.")
    print("(Optional: Runtime Change runtime type Sonata (Python 3.10) to fix)")
    PY = ENV_PYTHON
else:
    PY = sys.executable

# Quick sanity check: spconv works
result = subprocess.run(
    [PY, "-c",
     "import torch, spconv; "
     f"print(f'PyTorch {{torch.__version__}} CUDA {{torch.version.cuda}} spconv {{spconv.__version__}}'); "
     "x = torch.rand(1, 3, 64, 64, 64).cuda(); "
     "y = spconv.SubMConv3d(3, 3, 3, bias=False).cuda()(x); "
     f"print(f'spconv forward: {{y.shape}} OK')"],
    capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

# Compile custom CUDA ops (one-time)
%cd /content/voxel_group_classifier
for lib in ["pointops", "pointops2", "pointgroup_ops"]:
    print(f"Compiling {lib}...")
    r = subprocess.run(
        [PY, "-m", "pip", "install", f"libs/{lib}", "--quiet", "--no-build-isolation"],
        cwd="/content/voxel_group_classifier", capture_output=True, text=True)
    if r.returncode == 0:
        print(f"  {lib} OK")
    else:
        print(f"  {lib} FAILED: {r.stderr[-200:]}")

%cd /content/voxel_group_classifier
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

print("Environment ready!")

ModuleNotFoundError: No module named 'spconv'

In [3]:
# ═══════════════════════════════════════# CELL 3 — Download S3DIS + Convert# ═══════════════════════════════════════import numpy as np, h5py, glob, os, urllib.request, zipfileHDF5_URL = "https://huggingface.co/datasets/cminst/S3DIS/resolve/main/indoor3d_sem_seg_hdf5_data.zip"HDF5_DIR = "./indoor3d_sem_seg_hdf5_data"OUT_ROOT = "./data/s3dis"if not os.path.isdir(HDF5_DIR):    print("Downloading S3DIS (~1.7 GB)...")    urllib.request.urlretrieve(HDF5_URL, "/tmp/s3dis.zip")    with zipfile.ZipFile("/tmp/s3dis.zip") as zf:        zf.extractall(HDF5_DIR)h5_files = sorted(glob.glob(f"{HDF5_DIR}/*/*.h5"))print(f"Converting {len(h5_files)} HDF5 files...")for h5_path in h5_files:    fname = os.path.basename(h5_path).replace(".h5", "")    area_idx = int(fname.split("_")[-1])    area = f"Area_{area_idx}"    with h5py.File(h5_path, "r") as f:        data = f["data"][:]        labels = f["label"][:]    xyz = data[..., :3].reshape(-1, 3).astype(np.float32)    rgb = data[..., 3:6].reshape(-1, 3).astype(np.uint8)    seg = labels.reshape(-1, 1).astype(np.int16)    ins = np.zeros_like(seg, dtype=np.int16)    room_dir = os.path.join(OUT_ROOT, area, fname)    os.makedirs(room_dir, exist_ok=True)    np.save(os.path.join(room_dir, "coord.npy"), xyz)    np.save(os.path.join(room_dir, "color.npy"), rgb)    np.save(os.path.join(room_dir, "segment.npy"), seg)    np.save(os.path.join(room_dir, "instance.npy"), ins)print(f"Done. Data at {OUT_ROOT}/")

In [4]:
# IMI IMI IMI IMI IMI IMI IMI IMI IMI IMI IMI IMI
# CELL 4 - Train (Auto-Fallback)
# IMI IMI IMI IMI IMI IMI IMI IMI IMI IMI IMI IMI
import os, sys

ENV_PYTHON = "/usr/local/envs/pointcept-torch2.5.0-cu12.4/bin/python"
# Use conda python if kernel is not switched
PY = ENV_PYTHON if not os.path.exists(sys.executable + "/../conda-meta") and os.path.exists(ENV_PYTHON) else sys.executable
print(f"Using Python: {PY}")

MASKING_MODE = "cosine"  # "random" | "cosine" | "vgc"
EPOCHS = 10

if MASKING_MODE == "cosine":
    CONFIG = "configs/sonata/pretrain-sonata-v1m2-cosine-smoketest.py"
else:
    CONFIG = "configs/sonata/pretrain-sonata-v1m2-vgc-smoketest.py"

SAVE = f"exp/{MASKING_MODE}_conda_{EPOCHS}ep"
print(f"Config: {CONFIG} | Save: {SAVE}")

import subprocess
result = subprocess.run(
    [PY, "tools/train.py",
     "--config-file", CONFIG,
     "--options", f"save_path={SAVE}", f"epoch={EPOCHS}",
     "data.train.datasets.0.data_root=data/s3dis", "enable_wandb=False"],
    cwd="/content/voxel_group_classifier")
print(f"Exit code: {result.returncode}")


In [5]:
# ═══════════════════════════════════════# CELL 5 — Monitor with TensorBoard# ═══════════════════════════════════════%load_ext tensorboard%tensorboard --logdir exp --port 6006